In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

### Indexing (Document Ingestion)

In [ ]:
video_id = "-HzgcbRXUK8"
ytt_api = YouTubeTranscriptApi()

try:
    transcript = ytt_api.fetch(video_id)
    text = " ".join(item.text for item in transcript)
    print(text[:500])
except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")
except NoTranscriptFound:
    print("No transcript found for this video.")
except VideoUnavailable:
    print("Video is unavailable.")

- It's hard for us humans to make any kind of clean predictions about highly nonlinear, dynamical systems. But again, to your point, we might be very surprised
what classical learning systems might be able to do about even fluid. - Yes, exactly. I mean, fluid dynamics,
Navier-Stokes equations, these are traditionally thought of as very, very difficult intractable problems to do on classical systems. They take enormous amounts of compute, you know, weather prediction systems, you know, these kind


### Indexing (Text splitting)

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([text])

print(f"Total number of chunks: {len(chunks)}")
print(chunks[10])

Total number of chunks: 185
page_content='science kind of way? - Yeah, I think that there are actually a huge class of problems that could be couched in this way, the way we did AlphaGo and
the way we did AlphaFold, where, you know, you model what the
dynamics of the system is, the properties of that system, the environment that you
are trying to understand. And then that makes the
search for the solution or the prediction of
the next step efficient basically polynomial times, so tractable by a classical system, which a neural network is. It runs on normal computers, right, classical computers,
Turing machines in effect. And I think it's one of the most
interesting questions there is is how far can that paradigm go? You know, I think we've proven
the AI community in general that classical systems, Turing
machines can go a lot further than we previously thought. You know, they can do things like model the structures of proteins and play Go to better
than world champion level. And you kn

### Indexing (Embedding Generation and storing in Vector Store)

In [13]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [14]:
vector_store.index_to_docstore_id

{0: '1a158c8e-63ef-40d5-a7ed-fd8050749d08',
 1: '4ce32670-017d-44c5-aea1-1ae9c0c2f60e',
 2: 'b25d7c00-0171-42ad-a23f-269a9fd8a1d3',
 3: 'a307133b-8e21-44ca-925e-94738bc89c16',
 4: '7d47aed6-89aa-451b-a8c2-bc921f1365e8',
 5: 'a68a62a7-fb03-4f3b-9a3f-5935fd322523',
 6: '89aeccf3-be2e-4505-a2bb-8acad89a32f4',
 7: '79d212ad-f9b1-4c39-a98e-dac8ab1a13c6',
 8: 'fad0e3d4-0e01-4ca2-901e-d1f4d9ecaff4',
 9: '0db4c530-cac6-419a-8b0c-0e5a030b70a7',
 10: 'f7cf4e43-5bbf-475d-97b7-0f8718d9ccc2',
 11: 'fa0b0a0a-6b3d-45de-944d-d4aaa16548f2',
 12: '803e1aa8-4b45-46ed-b4d9-cf7a17c16d11',
 13: '2cfcdd78-e90d-4063-9a79-b6d555f4a45b',
 14: '6585e52e-e20d-4f6c-b4ec-e3dbdbaac1d2',
 15: '8e925ea5-d8f3-4b56-9775-afe6b5a5d33a',
 16: 'ada484fe-75a2-42f5-b2b5-5efd5dcb900f',
 17: 'd50d91bf-838c-4151-8a9a-6f6dd001b8af',
 18: '08092e47-28d8-46b2-b0a8-ef953f094c50',
 19: '5addcdeb-7446-48b3-9858-1103429386ca',
 20: '61fe895e-ea93-4b9f-a3e2-8603f29a1755',
 21: 'faa74eec-239b-414e-9159-5bf61c8f4d8d',
 22: '35918340-7bbb-

### Retriever

In [30]:
query = "What is Alphafold. And what about the nobel prize"

retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":4}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3405ea110>, search_kwargs={'k': 4})

In [31]:
retriever.invoke(query)

[Document(id='254bb413-59c8-4641-a76b-9d59ad46c01e', metadata={}, page_content="kind of equally valuable from an experiment that doesn't work. That should tell you if you've\ndesigned the experiment well and your hypotheses are are interesting, it should tell you a lot\nabout where to go next. And then you're effectively\ndoing a search process and using that information in, you know, very helpful ways. - So to go to your dream\nof modeling a cell, what are the big challenges\nthat lay ahead for us to make that happen? We should maybe highlight that AlphaFold, I mean there's just so many leaps. - Yeah.\n- So AlphaFold solved, if it's fair to say protein folding, and there's so many incredible things we could talk about there\nincluding the open sourcing, everything you've released. AlphaFold 3 is doing protein,\nRNA, DNA interactions, which is super complicated\nand fascinating. It's amenable to modeling. AlphaGenome predicts how\nsmall genetic changes. Like if we think about single mu

### Augmentation

In [32]:
prompt = PromptTemplate(
    template=""" 
    You are an helpfull assistant. 
    Answer only from the provided transcript context. 
    If the context is insufficient, just say you don't know.
    The context is:  
    {context}

    Question is: 
    {query}
    """
)

In [33]:
retriev_doc = retriever.invoke(query)

context_one_para = "\n\n".join(doc.page_content for doc in retriev_doc)

print(f"Context in one para: {context_one_para}")

Context in one para: kind of equally valuable from an experiment that doesn't work. That should tell you if you've
designed the experiment well and your hypotheses are are interesting, it should tell you a lot
about where to go next. And then you're effectively
doing a search process and using that information in, you know, very helpful ways. - So to go to your dream
of modeling a cell, what are the big challenges
that lay ahead for us to make that happen? We should maybe highlight that AlphaFold, I mean there's just so many leaps. - Yeah.
- So AlphaFold solved, if it's fair to say protein folding, and there's so many incredible things we could talk about there
including the open sourcing, everything you've released. AlphaFold 3 is doing protein,
RNA, DNA interactions, which is super complicated
and fascinating. It's amenable to modeling. AlphaGenome predicts how
small genetic changes. Like if we think about single mutations, how they link to actual function? So those, it seems like

-

In [34]:
final_prompt = prompt.invoke({'context': context_one_para, 'query': query})
final_prompt

StringPromptValue(text=" \n    You are an helpfull assistant. \n    Answer only from the provided transcript context. \n    If the context is insufficient, just say you don't know.\n    The context is:  \n    kind of equally valuable from an experiment that doesn't work. That should tell you if you've\ndesigned the experiment well and your hypotheses are are interesting, it should tell you a lot\nabout where to go next. And then you're effectively\ndoing a search process and using that information in, you know, very helpful ways. - So to go to your dream\nof modeling a cell, what are the big challenges\nthat lay ahead for us to make that happen? We should maybe highlight that AlphaFold, I mean there's just so many leaps. - Yeah.\n- So AlphaFold solved, if it's fair to say protein folding, and there's so many incredible things we could talk about there\nincluding the open sourcing, everything you've released. AlphaFold 3 is doing protein,\nRNA, DNA interactions, which is super complicat

### Generation

In [35]:
llm = ChatOpenAI()

final_answer = llm.invoke(final_prompt)
print(final_answer.content)

AlphaFold is a system that solved protein folding, and it also predicts how small genetic changes link to actual function. It is a project that showcases how AI can be used for the benefit of humanity. AlphaFold has made significant contributions to drug discovery and drug design acceleration. As for the Nobel Prize, there is no specific information in the provided context about AlphaFold and the Nobel Prize.


### Chain

In [39]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [36]:
prompt = PromptTemplate(
    template=""" 
    You are an helpfull assistant. 
    Answer only from the provided transcript context. 
    If the context is insufficient, just say you don't know.
    The context is:  
    {context}

    Question is: 
    {query}
    """
)

In [38]:
query = "What is Alphafold. And what about the nobel prize"

retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":4}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3405ea110>, search_kwargs={'k': 4})

In [40]:
def format_docs(retriev_doc):
    context_one_para = "\n\n".join(doc.page_content for doc in retriev_doc)
    return context_one_para

In [41]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'query' : RunnablePassthrough()
})

In [42]:
parallel_chain.invoke(query)

{'context': "kind of equally valuable from an experiment that doesn't work. That should tell you if you've\ndesigned the experiment well and your hypotheses are are interesting, it should tell you a lot\nabout where to go next. And then you're effectively\ndoing a search process and using that information in, you know, very helpful ways. - So to go to your dream\nof modeling a cell, what are the big challenges\nthat lay ahead for us to make that happen? We should maybe highlight that AlphaFold, I mean there's just so many leaps. - Yeah.\n- So AlphaFold solved, if it's fair to say protein folding, and there's so many incredible things we could talk about there\nincluding the open sourcing, everything you've released. AlphaFold 3 is doing protein,\nRNA, DNA interactions, which is super complicated\nand fascinating. It's amenable to modeling. AlphaGenome predicts how\nsmall genetic changes. Like if we think about single mutations, how they link to actual function? So those, it seems like\

In [43]:
parser = StrOutputParser()

In [45]:
main_chain = parallel_chain | prompt | llm | parser

main_chain.invoke(query)

"AlphaFold is a solution that solved protein folding, and it is able to model protein, RNA, DNA interactions. As for the Nobel Prize, I don't have information on that from the provided context."

## How to improve the RAG system: 

1. UI based enhancements: Make a website.

2. Evaluation
    * Ragas
    * LangSmith

3. Indexing
    * Document Ingestion.
    * Text Splitting. 
    * Vector Store. 

4. Retrieval 
    * Pre-Retrieval
        - Query rewriting using LLM. 
        - Multi-query generation.
        - Domain aware reouting. 

    * During Retrieval 
        - MMR
        - Hybrid Retrival
        - Reranking
    
    * Post-Retrieval
        - Contextual Compression

5. Augmentation
    * Prompt Templating
    * Answer Grounding
    * Contextual window optimization

6. Generation
    * Answer with citation
    * Guard railing